# Online Judge Problem Generation & Evaluation Pipeline
이 파이프라인은 50,000개의 문제를 생성하고, 모델을 사용하여 난이도를 평가한 뒤, 티어별로 정렬하여 zip으로 압축합니다.

In [ ]:
import os
import json
import random
import zipfile
from pathlib import Path
from vllm import LLM, SamplingParams

# --- 구성 설정 ---
MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
TOTAL_PROBLEMS = 50000
TIERS = ["브론즈", "실버", "골드", "플래티넘", "다이아몬드", "루비", "마스터"]
OUTPUT_DIR = Path("./final_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

# 1. 문제 생성 프롬프트 생성
def generate_manifest(count):
    problems = []
    for _ in range(count):
        problems.append({"task": "generate_algorithm_problem"})
    return problems

# 2. 난이도 평가 함수 (LLM 활용)
def evaluate_difficulty(problem_text, llm):
    prompt = f"이 알고리즘 문제의 난이도를 평가해줘. [브론즈, 실버, 골드, 플래티넘, 다이아몬드, 루비, 마스터] 중 하나와 1~5 (1이 가장 높음)로 답해줘. 형식: 티어 단계 (예: 골드 3)\n\n문제: {problem_text}"
    sampling_params = SamplingParams(max_tokens=10)
    output = llm.generate([prompt], sampling_params)
    return output[0].outputs[0].text.strip()

# 3. 파이프라인 실행
llm = LLM(model=MODEL_ID, tensor_parallel_size=4) # A100 80GB 기준
manifest = generate_manifest(TOTAL_PROBLEMS)
results = []

# (실제 생성 및 평가 루프... 대규모 작업은 배치 처리 권장)

# 4. 파일 정리 및 압축
def package_results(results):
    for prob in results:
        tier, level = prob['tier'], prob['level']
        dir_path = OUTPUT_DIR / tier / str(level)
        dir_path.mkdir(parents=True, exist_ok=True)
        with open(dir_path / f"{prob['id']}.md", "w") as f:
            f.write(prob['content'])
    
    with zipfile.ZipFile("oj_problems_50k.zip", 'w') as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for file in files:
                zf.write(os.path.join(root, file))

print("파이프라인 완료")